# Step 5: Baseline Models

In [31]:
# Load processed features and import the reusable splitting function

from pathlib import Path
import sys
import pandas as pd

current_folder = Path.cwd().resolve()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder

sys.path.append(str(project_root))

from src.data.data_splitting import create_data_splits

processed_features_path = project_root / "data" / "processed" / "url_features.csv"
df_features = pd.read_csv(processed_features_path)

feature_columns = [
    column for column in df_features.columns
    if column not in ["url", "label", "tld_type"]
]

print("Dataset loaded.")
print(f"Rows: {len(df_features):,}")
print(f"Numeric features: {len(feature_columns)}")

Dataset loaded.
Rows: 253,098
Numeric features: 27


In [32]:
# Convert the categorical TLD feature into one-hot columns

tld_dummies = pd.get_dummies(
    df_features["tld_type"],
    prefix="tld",
    dtype=int
)

df_model = pd.concat(
    [
        df_features.drop(columns=["tld_type"]),
        tld_dummies
    ],
    axis=1
)

all_feature_columns = [
    column for column in df_model.columns
    if column not in ["url", "label"]
]

print("TLD one-hot encoding completed.")
print(f"Total model features: {len(all_feature_columns)}")

TLD one-hot encoding completed.
Total model features: 548


In [33]:
# Create stratified training, validation, and test sets

(
    X_train,
    X_validation,
    X_test,
    y_train,
    y_validation,
    y_test
) = create_data_splits(
    dataframe=df_model,
    feature_columns=all_feature_columns,
    target_column="label",
    test_size=0.20,
    validation_size=0.20,
    random_state=42
)

print("Data splitting completed.")
print(f"Training rows: {len(X_train):,}")
print(f"Validation rows: {len(X_validation):,}")
print(f"Test rows: {len(X_test):,}")

print("\nTraining label distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nValidation label distribution:")
print(y_validation.value_counts(normalize=True).round(4))

print("\nTest label distribution:")
print(y_test.value_counts(normalize=True).round(4))

Data splitting completed.
Training rows: 151,858
Validation rows: 50,620
Test rows: 50,620

Training label distribution:
label
0    0.5113
1    0.4887
Name: proportion, dtype: float64

Validation label distribution:
label
0    0.5113
1    0.4887
Name: proportion, dtype: float64

Test label distribution:
label
0    0.5113
1    0.4887
Name: proportion, dtype: float64


In [34]:
# Save split data for reproducible baseline-model experiments

split_folder = project_root / "data" / "processed" / "splits"
split_folder.mkdir(parents=True, exist_ok=True)

X_train.to_csv(split_folder / "X_train.csv", index=False)
X_validation.to_csv(split_folder / "X_validation.csv", index=False)
X_test.to_csv(split_folder / "X_test.csv", index=False)

y_train.to_csv(split_folder / "y_train.csv", index=False)
y_validation.to_csv(split_folder / "y_validation.csv", index=False)
y_test.to_csv(split_folder / "y_test.csv", index=False)

print("All six split files saved to:")
print(split_folder)

All six split files saved to:
C:\Users\Jiya\Documents\Malicious URL Detection\data\processed\splits


In [35]:
# Create a leakage-safe preprocessing and Logistic Regression pipeline

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

print("Logistic Regression pipeline created.")

Logistic Regression pipeline created.


In [36]:
# Train Logistic Regression using training data only

logistic_pipeline.fit(X_train, y_train)

print("Logistic Regression training completed.")

Logistic Regression training completed.


In [37]:
# Evaluate Logistic Regression on the validation set

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

logistic_validation_predictions = logistic_pipeline.predict(X_validation)

logistic_validation_metrics = {
    "model": "Logistic Regression",
    "split": "validation",
    "accuracy": accuracy_score(y_validation, logistic_validation_predictions),
    "precision": precision_score(y_validation, logistic_validation_predictions),
    "recall": recall_score(y_validation, logistic_validation_predictions),
    "f1_score": f1_score(y_validation, logistic_validation_predictions)
}

print(logistic_validation_metrics)

print("\nConfusion matrix:")
print(confusion_matrix(y_validation, logistic_validation_predictions))

{'model': 'Logistic Regression', 'split': 'validation', 'accuracy': 0.8876333465033583, 'precision': 0.9039701391245334, 'recall': 0.8615782664941786, 'f1_score': 0.8822652757078987}

Confusion matrix:
[[23620  2264]
 [ 3424 21312]]


In [38]:
# Save the trained Logistic Regression pipeline

import joblib

models_folder = project_root / "models"
models_folder.mkdir(parents=True, exist_ok=True)

logistic_model_path = models_folder / "logistic_regression_pipeline.joblib"

joblib.dump(logistic_pipeline, logistic_model_path)

print("Logistic Regression model saved to:")
print(logistic_model_path)

Logistic Regression model saved to:
C:\Users\Jiya\Documents\Malicious URL Detection\models\logistic_regression_pipeline.joblib


# Random Forest

In [39]:
# Import and create the Random Forest baseline model

from sklearn.ensemble import RandomForestClassifier

random_forest_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    (
        "model",
        RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

print("Random Forest pipeline created.")

Random Forest pipeline created.


In [40]:
# Train Random Forest on training data only

random_forest_pipeline.fit(X_train, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [41]:
# Evaluate Random Forest on the validation set

random_forest_validation_predictions = (
    random_forest_pipeline.predict(X_validation)
)

random_forest_validation_metrics = {
    "model": "Random Forest",
    "split": "validation",
    "accuracy": accuracy_score(
        y_validation,
        random_forest_validation_predictions
    ),
    "precision": precision_score(
        y_validation,
        random_forest_validation_predictions
    ),
    "recall": recall_score(
        y_validation,
        random_forest_validation_predictions
    ),
    "f1_score": f1_score(
        y_validation,
        random_forest_validation_predictions
    )
}

print(random_forest_validation_metrics)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_validation,
        random_forest_validation_predictions
    )
)

{'model': 'Random Forest', 'split': 'validation', 'accuracy': 0.966910312129593, 'precision': 0.9762504646648218, 'recall': 0.9555304010349288, 'f1_score': 0.9657793123174045}

Confusion matrix:
[[25309   575]
 [ 1100 23636]]


In [42]:
# Save the trained Random Forest pipeline

random_forest_model_path = (
    models_folder / "random_forest_pipeline.joblib"
)

joblib.dump(
    random_forest_pipeline,
    random_forest_model_path
)

print("Random Forest model saved to:")
print(random_forest_model_path)

Random Forest model saved to:
C:\Users\Jiya\Documents\Malicious URL Detection\models\random_forest_pipeline.joblib


In [43]:
# Install XGBoost into the active notebook environment

import sys
!{sys.executable} -m pip install xgboost

In [44]:
# Import and create the XGBoost baseline model

from xgboost import XGBClassifier

xgboost_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    (
        "model",
        XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )
    )
])

print("XGBoost pipeline created.")

XGBoost pipeline created.


In [45]:
# Train XGBoost on training data only

xgboost_pipeline.fit(X_train, y_train)

print("XGBoost training completed.")

XGBoost training completed.


In [46]:
# Evaluate XGBoost on the validation set

xgboost_validation_predictions = (
    xgboost_pipeline.predict(X_validation)
)

xgboost_validation_metrics = {
    "model": "XGBoost",
    "split": "validation",
    "accuracy": accuracy_score(
        y_validation,
        xgboost_validation_predictions
    ),
    "precision": precision_score(
        y_validation,
        xgboost_validation_predictions
    ),
    "recall": recall_score(
        y_validation,
        xgboost_validation_predictions
    ),
    "f1_score": f1_score(
        y_validation,
        xgboost_validation_predictions
    )
}

print(xgboost_validation_metrics)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_validation,
        xgboost_validation_predictions
    )
)

{'model': 'XGBoost', 'split': 'validation', 'accuracy': 0.9654879494271039, 'precision': 0.9803184154444027, 'recall': 0.9484152652005174, 'f1_score': 0.9641029855959891}

Confusion matrix:
[[25413   471]
 [ 1276 23460]]


In [47]:
# Save the trained XGBoost pipeline

xgboost_model_path = models_folder / "xgboost_pipeline.joblib"

joblib.dump(
    xgboost_pipeline,
    xgboost_model_path
)

print("XGBoost model saved to:")
print(xgboost_model_path)

XGBoost model saved to:
C:\Users\Jiya\Documents\Malicious URL Detection\models\xgboost_pipeline.joblib


In [48]:
# Save baseline validation metrics for model comparison

baseline_results = pd.DataFrame([
    logistic_validation_metrics,
    random_forest_validation_metrics,
    xgboost_validation_metrics
])

metrics_folder = project_root / "results" / "metrics"
metrics_folder.mkdir(parents=True, exist_ok=True)

baseline_results_path = metrics_folder / "baseline_validation_metrics.csv"

baseline_results.to_csv(baseline_results_path, index=False)

print("Baseline validation results:")
display(baseline_results)

print("\nSaved to:")
print(baseline_results_path)

Baseline validation results:


,model,split,accuracy,precision,recall,f1_score
0,Logistic Regression,validation,0.887633,0.903970,0.861578,0.882265
1,Random Forest,validation,0.966910,0.976250,0.955530,0.965779
2,XGBoost,validation,0.965488,0.980318,0.948415,0.964103



Saved to:
C:\Users\Jiya\Documents\Malicious URL Detection\results\metrics\baseline_validation_metrics.csv
